#SVD FlagOS

# The Math: What is SVD?

At its core, SVD is a way to break down any complex matrix (like a layer of weights in an LLM or an image) into three simpler, foundational matrices.If you have a matrix $A$, SVD factors it into:$$A = U \Sigma V^T$$

* $V^T$ (Right Singular Vectors): A rotation matrix. It rotates your data.

* $\Sigma$ (Singular Values): A diagonal scaling matrix. It stretches or squishes the data along the axes. This tells you which features are the most "important" (the biggest numbers).

* $U$ (Left Singular Vectors): Another rotation matrix. It rotates the data into its final position.

Every single linear transformation in the universe can be broken down into a rotate, a stretch, and another rotate.

---

# Why is SVD a "Difficult" Tier Operator in Triton?

Writing a basic math operator in Triton is easy because we apply the math to each number independently. SVD is notoriously hard on GPUs for three reasons:

1. Iterative: There is no simple algebraic formula to calculate SVD for large matrices. Algorithms (like the Jacobi method or Golub-Kahan bidiagonalization) require looping, estimating, and refining. GPUs are designed for parallel tasks, not looping through branches.

2. Memory Bottlenecks: Because it requires constantly referencing and updating entire rows and columns of matrices, managing the GPU's extremely limited Shared Random Access Memory (SRAM) perfectly. If we write to the slower global memory too often, the speedup ratio will completely fail the $\ge 0.9$ benchmark.

3. Numerical Stability: Floating-point rounding errors stack up incredibly fast during iterative matrix decomposition. Matching PyTorch's native precision (torch.allclose) will require deep knowledge of handling floating-point arithmetic.

##1: Environment Setup

In [1]:
# Install the required libraries
!pip install -q torch triton

##2: Imports & Device Check

In [2]:
import torch
import triton
import triton.language as tl
import time

# Verify the Colab environment has a GPU attached
assert torch.cuda.is_available(), "GPU is not enabled! Go to Runtime -> Change runtime type and select T4 GPU."
print(f"Active Device: {torch.cuda.get_device_name(0)}")

Active Device: Tesla T4


##3: The Triton Kernel Skeleton

In [84]:
@triton.jit
def svd_jacobi_kernel(
    a_ptr, u_ptr, s_ptr, vt_ptr,
    M, N,
    stride_am, stride_an,
    stride_um, stride_un,
    stride_sm,
    stride_vtm, stride_vtn,
    BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr,
    SUB_BLOCK_SIZE: tl.constexpr = 16
):
    pid = tl.program_id(axis=0)

    offs_m = tl.arange(0, BLOCK_SIZE_M)
    offs_n = tl.arange(0, BLOCK_SIZE_N)

    a_ptrs = a_ptr + (offs_m[:, None] * stride_am + offs_n[None, :] * stride_an)
    mask_2d = (offs_m[:, None] < M) & (offs_n[None, :] < N)

    # 1. Load entire matrix into registers
    a_block = tl.load(a_ptrs, mask=mask_2d, other=0.0)
    v_block = tl.where(offs_m[:, None] == offs_n[None, :], 1.0, 0.0)

    NUM_SWEEPS = 10
    TOLERANCE = 1e-5

    # 1. Setup dynamic loop variables
    sweep = 0
    converged = 0 # Triton prefers integers for booleans in loops

    # 2. Use a dynamic while loop instead of a static for loop
    while (sweep < NUM_SWEEPS) and (converged == 0):

      for p_start in range(0, N, SUB_BLOCK_SIZE):
            for q_start in range(p_start + SUB_BLOCK_SIZE, N, SUB_BLOCK_SIZE):

                  # ... inside your p_start / q_start loop ...

                  # 1. Pre-compute the Gram matrix ONCE per block pairing
                  g_block = tl.dot(tl.trans(a_block), a_block)

                  # 2. Initialize an Identity matrix to accumulate our block rotations
                  R_acc = tl.where(offs_m[:, None] == offs_n[None, :], 1.0, 0.0)

                  # 3. Calculate all rotations for this block pairing
                  for i in range(SUB_BLOCK_SIZE):
                      p = p_start + i
                      q = q_start + i

                      if p < N and q < N:
                          # Extract directly from the Gram matrix without full-block masking
                          alpha = tl.sum(tl.where((offs_m[:, None] == p) & (offs_n[None, :] == p), g_block, 0.0))
                          beta  = tl.sum(tl.where((offs_m[:, None] == q) & (offs_n[None, :] == q), g_block, 0.0))
                          gamma = tl.sum(tl.where((offs_m[:, None] == p) & (offs_n[None, :] == q), g_block, 0.0))

                          if tl.abs(gamma) > 1e-6:
                              zeta = (beta - alpha) / (2.0 * gamma + 1e-8)
                              t = tl.where(zeta > 0, 1.0 / (zeta + tl.sqrt(1.0 + zeta*zeta)),
                                                    -1.0 / (-zeta + tl.sqrt(1.0 + zeta*zeta)))
                              c = 1.0 / tl.sqrt(1.0 + t*t)
                              s = t * c

                              # Update ONLY the accumulator matrix (R_acc), not the whole A matrix
                              # We update the 4 intersecting points for p and q
                              R_acc = tl.where((offs_m[:, None] == p) & (offs_n[None, :] == p), c, R_acc)
                              R_acc = tl.where((offs_m[:, None] == q) & (offs_n[None, :] == q), c, R_acc)
                              R_acc = tl.where((offs_m[:, None] == q) & (offs_n[None, :] == p), -s, R_acc)
                              R_acc = tl.where((offs_m[:, None] == p) & (offs_n[None, :] == q), s, R_acc)

                              # Update the Gram matrix directly so future iterations in this block are accurate
                              # G = R^T * G * R
                              g_block = tl.dot(tl.trans(R_acc), tl.dot(g_block, R_acc))

                  # 4. ONE massive Tensor Core update for the entire block pairing
                  a_block = tl.dot(a_block, R_acc)
                  v_block = tl.dot(v_block, R_acc)

            # 4. Early Stopping Check
            g_final = tl.dot(tl.trans(a_block), a_block)
            off_diag = tl.where(offs_m[:, None] == offs_n[None, :], 0.0, g_final)
            error = tl.sum(tl.abs(off_diag))

            if error < TOLERANCE:
                converged = 1 # Triggers loop exit on the next evaluation

            sweep += 1

    # Final Sigma, U, and write-back
    s_block = tl.sqrt(tl.sum(a_block * a_block, axis=0))
    u_block = a_block / (s_block[None, :] + 1e-8)
    vt_block = tl.trans(v_block)

    tl.store(u_ptr + (offs_m[:, None] * stride_um + offs_n[None, :] * stride_un), u_block, mask=mask_2d)
    tl.store(s_ptr + (offs_n * stride_sm), s_block, mask=offs_n < N)
    tl.store(vt_ptr + (offs_m[:, None] * stride_vtm + offs_n[None, :] * stride_vtn), vt_block, mask=mask_2d)

##4: The Python/PyTorch Wrapper

In [85]:
def run_custom_svd(A):
    M, N = A.shape

    # Allocate output tensors inside GPU memory
    U = torch.empty((M, M), device=A.device, dtype=A.dtype)
    S = torch.empty((N,), device=A.device, dtype=A.dtype)
    Vt = torch.empty((N, N), device=A.device, dtype=A.dtype)

    # Align blocks to powers of 2 for memory alignment vectorization
    BLOCK_SIZE_M = triton.next_power_of_2(M)
    BLOCK_SIZE_N = triton.next_power_of_2(N)

    grid = (1,)

    # Launch Kernel passing all base pointers and layout strides
    svd_jacobi_kernel[grid](
        A, U, S, Vt,
        M, N,
        A.stride(0), A.stride(1),
        U.stride(0), U.stride(1),
        S.stride(0),
        Vt.stride(0), Vt.stride(1),
        BLOCK_SIZE_M=BLOCK_SIZE_M,
        BLOCK_SIZE_N=BLOCK_SIZE_N
    )

    return U, S, Vt

##5: The Competition Benchmark Harness

In [88]:
def benchmark_svd_table():
    # Define up to 3 shapes to benchmark
    # Note: Kept relatively small (16, 32, 64) to fit inside the current SRAM limits
    test_shapes = [(16, 16), (32, 32), (64, 64)]

    # 1. Print the Table Header
    header = f"{'Shape':<15} | {'Dimension':<12} | {'Triton (μs)':<15} | {'PyTorch (μs)':<15} | {'Speedup':<10}"
    separator = "-" * len(header)

    print("\n--- SVD Benchmark Results ---")
    print(separator)
    print(header)
    print(separator)

    # 2. Loop through each shape and run the benchmarker
    for M, N in test_shapes:
        A = torch.randn((M, N), device='cuda', dtype=torch.float32)

        # Define the lambda functions for the benchmarker
        native_fn = lambda: torch.linalg.svd(A, full_matrices=True)
        custom_fn = lambda: run_custom_svd(A)

        # Benchmarking (returns milliseconds)
        # Suppressing stdout here just in case, to keep the table clean
        native_ms = triton.testing.do_bench(native_fn, warmup=25, rep=100)
        custom_ms = triton.testing.do_bench(custom_fn, warmup=25, rep=100)

        # Convert milliseconds (ms) to microseconds (μs)
        native_us = native_ms * 1000
        custom_us = custom_ms * 1000

        # Calculate True Speedup Ratio
        speedup = native_us / custom_us

        # Format strings for the table
        shape_str = f"({M}, {N})"
        dim_str = f"{M * N}"  # Total elements (or you can change this to "2" if Dimension meant Rank)

        # Print the formatted row
        row = f"{shape_str:<15} | {dim_str:<12} | {custom_us:<15.2f} | {native_us:<15.2f} | {speedup:<10.2f}"
        print(row)

    print(separator)
    print("\nNote: Accuracy validation is skipped in this loop for cleaner output. Ensure accuracy passes on a single shape first.")

# Run the table benchmark
benchmark_svd_table()


--- SVD Benchmark Results ---
-------------------------------------------------------------------------------
Shape           | Dimension    | Triton (μs)     | PyTorch (μs)    | Speedup   
-------------------------------------------------------------------------------
(16, 16)        | 256          | 23.43           | 717.18          | 30.61     
(32, 32)        | 1024         | 126.18          | 485.45          | 3.85      
(64, 64)        | 4096         | 9555.48         | 1482.86         | 0.16      
-------------------------------------------------------------------------------

Note: Accuracy validation is skipped in this loop for cleaner output. Ensure accuracy passes on a single shape first.
